# 1 Initialize the Database

All the code related to data management is in the `EnvironmentData` class. This makes life easier - for example: we can send the CatsUserID once and it becomes a class property. Then, when we call other operations we don't have to send this information again.

When you create a new instance of `EnvironmentData` and there is no database, it will pull historical data and initialize the database. 

In [1]:
# Clear prior data. 
import os, sys, shutil

# Add parent directory to Python path to import EnvironmentData.
sys.path.append(os.path.dirname(os.getcwd()))

# Get the EnvironmentData class.
from EnvironmentData import EnvironmentData 

# The project adds to existing data so we need to clear that data to get a solid test from scratch.
if os.path.exists('../data'):
    shutil.rmtree('../data')
    os.makedirs('../data')

# Initialize EnvironmentData. This will run the historical data pull.
envdt = EnvironmentData(
    #days_back = 365 * 2,
    days_back = 7,
    coris_enabled = True,
    licor_enabled = True,
    conserv_enabled = True, 
    testing = True,
    #testing = False,
    # Since we are running from the experiments/ folder, we need to tell the class to use the parent directory as home.
    home_directory = ".."
)

DEBUG: Enabled data sources: ['Conserv', 'Coris', 'LI-COR']


Gathering LI-COR readings: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  3.28it/s]


Detailed information is saved in the log:

In [2]:
# Detailed info is saved in the log.
with open('../data/EnvironmentData.log', 'r') as file:
    for line in file.read().splitlines()[:10]:
        print(line)

2025-12-03 12:43:38,795 - EnvironmentData - INFO - Initialized Conserv client with 5 customers
2025-12-03 12:43:38,796 - EnvironmentData - INFO - Enabled data sources: ['Conserv', 'Coris', 'LI-COR']
2025-12-03 12:43:38,797 - EnvironmentData - INFO - Fetching Conserv historical data for all customers
2025-12-03 12:43:38,797 - EnvironmentData - INFO - Fetching Conserv data for period: 1764186218 to 1764791018
2025-12-03 12:43:38,797 - EnvironmentData - INFO - Running in test mode - only processing first customer: 333
2025-12-03 12:43:38,811 - EnvironmentData - INFO - Fetching data for customer 333
2025-12-03 12:43:38,811 - EnvironmentData - INFO - Exporting chunk for customer 333: 2025-11-26 19:43:38+00:00 to 2025-12-03 19:43:38+00:00
2025-12-03 12:43:38,812 - EnvironmentData - INFO - Starting export for customer 333: 2025-11-26 19:43:38+00:00 to 2025-12-03 19:43:38+00:00
2025-12-03 12:43:38,812 - EnvironmentData - INFO - Conserv API POST https://api.conserv.io/v1/sensors/export headers=

This saves our intermediate data to `data/sensor_readings.parquet`. 

Initially, we leave the data mostly as-is. We'll clean, add formatted dates, consolidate readings from the same device, etc. when moving to analytical steps, this preserves the source data so we can always change our mind later about how we decide to view it. 

However, at this point we are taking care to standardize the data format between different API sources. 

There are just a few columns because this is only historical data. We'll bring in current data shortly, and that will add more columns. 

In [3]:
import polars
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "Coris").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1764186218,1764791132,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.790001,null,true
1764187118,1764791132,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.879997,null,true
1764188018,1764791132,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.760002,null,true
1764188918,1764791132,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.720001,null,true
1764189818,1764791132,"""Coris""","""coris:12162""","""Peabody TH-L Mammal Hall D444""","""coris:21373""","""PYPM__0100104SET____ Temp YPM …","""Temperature""",68.699997,null,true


In [4]:
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "LI-COR").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1764186300,1764791145,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""","""RX Station 1_Temperature""","""Temperature""",70.412308,null,true
1764187200,1764791145,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""","""RX Station 1_Temperature""","""Temperature""",70.296478,null,true
1764188100,1764791145,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""","""RX Station 1_Temperature""","""Temperature""",70.142036,null,true
1764189000,1764791145,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""","""RX Station 1_Temperature""","""Temperature""",70.064812,null,true
1764189900,1764791145,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179174-1""","""RX Station 1_Temperature""","""Temperature""",70.064812,null,true


In [5]:
polars.read_parquet('../data/sensor_readings.parquet').filter(polars.col("Source") == "Conserv").head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,Historical
i64,i32,str,str,str,str,str,str,f32,f32,bool
1764186437,1764791019,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""conserv:333:c009096:Temperatur…","""Temperature""",69.494003,null,true
1764187337,1764791019,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""conserv:333:c009096:Temperatur…","""Temperature""",69.350006,null,true
1764188237,1764791019,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""conserv:333:c009096:Temperatur…","""Temperature""",69.169998,null,true
1764189137,1764791019,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""conserv:333:c009096:Temperatur…","""Temperature""",69.098007,null,true
1764190037,1764791019,"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:Temperatur…","""conserv:333:c009096:Temperatur…","""Temperature""",69.115997,null,true


# 2 Get Current Readings

Now we can start gathering and appending readings. There is a function `get_current_readings` that is run throughout the day, every 10 minutes for example. This function creates a parquet file at `data/new-readings` with the UTC as a filename. At the end of the day, all these readings will be consolidated into the database. 

Here is a sample of the readings:

In [6]:
# Wait 15 minutes to allow a new Conserv reading.
# import time
# time.sleep(15 * 60)  # Wait 15 minutes (900 seconds)

# envdt.get_current_readings()

# # Data is read into new-readings folder for consolidation at the end of the day.
# import os
# filename = os.listdir('../data/new-readings')[0]
# print(filename)
# polars.read_parquet('../data/new-readings/' + filename).sample(5)

# 3 Consolidate Readings

At the end of the day, new readings will be consolidated into the table. At the same time, the analytical tables will be generated. 

Analytical tables include:

* `device_readings.parquet`: Sensor readings reorganized to one row per Device and UTC, with measurements across columns vs measurements across rows.* 
* `sensors.parquet`: Information about the unique sensors. Includes information extracted from SensorName. Join this to Sensors during analysis to enhance with Building, Room, Direction, etc.
* `devices.parquet`: Information about unique devices. Includes information extracted from SensorName. 
* `utcs.parquet`: Information related to the UTC times in various datasets. Join to Sensors or Devices to enhance with Date, Time, Year, Hour, Weekday, etc.
* `sensor_readings_daily.parquet`: Example of sensor readings summarized to the daily level which reduces row count by 99.3% for even faster queries.
* `device_readings_daily.parquet`: Example of device readings summarized to the daily level which reduces row count by 99.3% for even faster queries. 

We fully re-generate analytical tables during each consolidation. The data is small enough that this is a fairly quick process, so re-running it in full each time will make it easy to ensure consistency as we expand and change the project. 

In [7]:
# To consolidate these into the database, run consolidate_readings.
envdt.consolidate_readings()

# New-readings files are gone now.
# They get deleted each day to confirm that they have been loaded into the database and prepare for the next consolidation.
if os.path.exists('../data/new-readings'):
    print(os.listdir('../data/new-readings'))

In [8]:
# Use this to re-run if you change the consolidation code.

# from EnvironmentData import EnvironmentData 
# envdt = EnvironmentData(
#     #days_back = 365 * 2,
#     days_back = 7,
#     coris_enabled = True,
#     licor_enabled = True,
#     conserv_enabled = True, 
#     #testing = True,
#     testing = False,
#     # Since we are running from the experiments/ folder, we need to tell the class to use the parent directory as home.
#     home_directory = ".."
# )
# envdt.close()
# del envdt

**^^ We want this to be empty** since we have consolidated new readings into the historical data. 

Once we are done working with data intake/processing, we close the class to release the file lock on the log file.

In [9]:
# When done, close the connection to the logs. 
envdt.close()

Let's look at the data we have now:

In [10]:
# Sensor Readings
# The first rows will be missing the extra fields like HexGatewayMac, etc.
#   I am pulling in some extra fields like DeviceID and DeviceName so we have that by historical. 
#   But some don't make sense to  backfill so they'll be null.
sensor_readings = polars.read_parquet('../data/sensor_readings.parquet')
sensor_readings.head()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,SensorReadingUTC_SecondsFromPrior,Historical
i64,i32,str,str,str,str,str,str,f32,f32,i64,bool
1764187035,1764791019,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""conserv:333:c008706:RH""","""RH""",null,48.599998,null,true
1764187935,1764791019,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""conserv:333:c008706:RH""","""RH""",null,48.25,null,true
1764188835,1764791019,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""conserv:333:c008706:RH""","""RH""",null,48.279999,null,true
1764189735,1764791019,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""conserv:333:c008706:RH""","""RH""",null,48.439999,null,true
1764190635,1764791019,"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH""","""conserv:333:c008706:RH""","""RH""",null,48.52,null,true


In [11]:
# Recent rows will have the full data, aside from nulls due to a sensor not providing a reading type.
sensor_readings.tail()

SensorReadingUTC,QueryUTC,Source,DeviceID,DeviceName,SensorID,SensorName,SensorType,SensorReadingF,SensorReadingRh,SensorReadingUTC_SecondsFromPrior,Historical
i64,i32,str,str,str,str,str,str,f32,f32,i64,bool
1764783900,1764791145,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",71.763672,null,null,true
1764784800,1764791145,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",71.72506,null,null,true
1764785700,1764791145,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",71.686447,null,null,true
1764786600,1764791145,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",71.802277,null,null,true
1764787500,1764791145,"""LI-COR""","""licor:22202142""","""RX Station 1""","""licor:22202142-22179175-1""","""RX Station 1_Temperature""","""Temperature""",71.840889,null,null,true


In [12]:
# Device Readings.
device_readings = polars.read_parquet('../data/device_readings.parquet')
device_readings.head()

Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,SensorReadingUTC,QueryUTC,Historical,SensorReadingF,SensorReadingRh
str,str,str,str,str,str,i64,i32,bool,f32,f32
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH | conse…","""conserv:333:c008706:RH | conse…","""RH | Temperature""",1764187035,1764791019,true,69.242004,48.599998
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH | conse…","""conserv:333:c008706:RH | conse…","""RH | Temperature""",1764187935,1764791019,true,69.350006,48.25
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH | conse…","""conserv:333:c008706:RH | conse…","""RH | Temperature""",1764188835,1764791019,true,69.332001,48.279999
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH | conse…","""conserv:333:c008706:RH | conse…","""RH | Temperature""",1764189735,1764791019,true,69.296005,48.439999
"""Conserv""","""conserv:333:c008706""","""BYCBA_0400410__N____""","""conserv:333:c008706:RH | conse…","""conserv:333:c008706:RH | conse…","""RH | Temperature""",1764190635,1764791019,true,69.061996,48.52


In [13]:
# Sensors
sensors = polars.read_parquet('../data/sensors.parquet')
sensors.head()

Source,SensorID,SensorName,SensorType,DeviceID
str,str,str,str,str
"""Conserv""","""conserv:333:c008706:RH""","""conserv:333:c008706:RH""","""RH""","""conserv:333:c008706"""
"""Conserv""","""conserv:333:c008706:Temperatur…","""conserv:333:c008706:Temperatur…","""Temperature""","""conserv:333:c008706"""
"""Conserv""","""conserv:333:c008733:RH""","""conserv:333:c008733:RH""","""RH""","""conserv:333:c008733"""
"""Conserv""","""conserv:333:c008733:Temperatur…","""conserv:333:c008733:Temperatur…","""Temperature""","""conserv:333:c008733"""
"""Conserv""","""conserv:333:c008734:RH""","""conserv:333:c008734:RH""","""RH""","""conserv:333:c008734"""


In [14]:
# Devices. 
devices = polars.read_parquet('../data/devices.parquet')
devices.head()

Source,DeviceID,DeviceName,SensorIDs,SensorNames,SensorTypes,DeviceSerialFromName,BuildingID,Building,Room,CardinalDirection
str,str,str,str,str,str,str,str,str,str,str
"""Conserv""","""conserv:333:c009072""","""BBARCH0100001_______""","""conserv:333:c009072:RH | conse…","""conserv:333:c009072:RH | conse…","""RH | Temperature""",null,null,null,null,null
"""Conserv""","""conserv:333:c009073""","""BBARCHB100116_______""","""conserv:333:c009073:RH | conse…","""conserv:333:c009073:RH | conse…","""RH | Temperature""",null,null,null,null,null
"""Conserv""","""conserv:333:c009081""","""BCSC__01H103________""","""conserv:333:c009081:RH | conse…","""conserv:333:c009081:RH | conse…","""RH | Temperature""",null,null,null,null,null
"""Conserv""","""conserv:333:c009096""","""BFAST__BS___________""","""conserv:333:c009096:RH | conse…","""conserv:333:c009096:RH | conse…","""RH | Temperature""",null,null,null,null,null
"""Conserv""","""conserv:333:c008924""","""BYCBA_0300318_______""","""conserv:333:c008924:RH | conse…","""conserv:333:c008924:RH | conse…","""RH | Temperature""",null,null,null,null,null


In [15]:
# UTC Date/Time Info
utcs = polars.read_parquet('../data/utcs.parquet').head()
utcs.head()

UTC,datetime_utc,datetime_est,date,time,year,month,day_of_week,day_of_week_monday1_sunday7,hour_24,hour_12,am_pm
i64,datetime[μs],"datetime[μs, America/New_York]",date,time,i32,i8,str,i8,i8,i8,str
1764360204,2025-11-28 13:03:24,2025-11-28 08:03:24 EST,2025-11-28,08:03:24,2025,11,"""Friday""",5,8,8,"""AM"""
1764229133,2025-11-27 00:38:53,2025-11-26 19:38:53 EST,2025-11-26,19:38:53,2025,11,"""Wednesday""",3,19,7,"""PM"""
1764622350,2025-12-01 13:52:30,2025-12-01 08:52:30 EST,2025-12-01,08:52:30,2025,12,"""Monday""",1,8,8,"""AM"""
1764229143,2025-11-27 00:39:03,2025-11-26 19:39:03 EST,2025-11-26,19:39:03,2025,11,"""Wednesday""",3,19,7,"""PM"""
1764753439,2025-12-03 02:17:19,2025-12-02 21:17:19 EST,2025-12-02,21:17:19,2025,12,"""Tuesday""",2,21,9,"""PM"""


In [16]:
# Daily Sensor Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
sensor_readings_daily = polars.read_parquet('../data/sensor_readings_daily.parquet')
sensor_readings_daily.head()

Source,date,SensorID,row_count,SensorReadingF_sum,SensorReadingRh_sum,SensorReadingF_min,SensorReadingRh_min,SensorReadingF_max,SensorReadingRh_max
str,date,str,u32,f32,f32,f32,f32,f32,f32
"""Conserv""",2025-12-03,"""conserv:333:c008706:RH""",1,0.0,47.84,null,47.84,null,47.84
"""Conserv""",2025-12-03,"""conserv:333:c008706:Temperatur…",1,70.951996,0.0,70.951996,null,70.951996,null
"""Conserv""",2025-12-03,"""conserv:333:c008789:RH""",1,0.0,43.630001,null,43.630001,null,43.630001
"""Conserv""",2025-12-03,"""conserv:333:c008789:Temperatur…",1,70.736008,0.0,70.736008,null,70.736008,null
"""Conserv""",2025-12-03,"""conserv:333:c008903:RH""",1,0.0,45.09,null,45.09,null,45.09


In [17]:
# Daily Device Readings
# Averages are calculated by summing the "sum" and "row_count" columns and dividing to get the average. 
device_readings_daily = polars.read_parquet('../data/device_readings_daily.parquet')
device_readings_daily.head()

Source,date,DeviceID,row_count,SensorReadingF_sum,SensorReadingRh_sum,SensorReadingF_min,SensorReadingRh_min,SensorReadingF_max,SensorReadingRh_max
str,date,str,u32,f32,f32,f32,f32,f32,f32
"""Conserv""",2025-12-03,"""conserv:333:c008706""",1,70.951996,47.84,70.951996,47.84,70.951996,47.84
"""Conserv""",2025-12-03,"""conserv:333:c008789""",1,70.736008,43.630001,70.736008,43.630001,70.736008,43.630001
"""Conserv""",2025-12-03,"""conserv:333:c008903""",1,69.422005,45.09,69.422005,45.09,69.422005,45.09
"""Conserv""",2025-12-03,"""conserv:333:c008943""",1,69.494003,44.860001,69.494003,44.860001,69.494003,44.860001
"""Conserv""",2025-12-03,"""conserv:333:c008965""",1,69.944,46.040001,69.944,46.040001,69.944,46.040001


In [18]:
# Differentiate historical vs. cron readings by filtering on Historical = true.
import duckdb
duckdb.sql("""
    SELECT *
    FROM read_parquet('../data/device_readings.parquet') 
    WHERE Historical
    LIMIT 5
""").to_df()

,Source,DeviceID,DeviceName,Sensors,SensorNames,SensorTypes,SensorReadingUTC,QueryUTC,Historical,SensorReadingF,SensorReadingRh
0,Conserv,conserv:333:c008706,BYCBA_0400410__N____,conserv:333:c008706:RH | conserv:333:c008706:T...,conserv:333:c008706:RH | conserv:333:c008706:T...,RH | Temperature,1764187035,1764791019,True,69.242004,48.599998
1,Conserv,conserv:333:c008706,BYCBA_0400410__N____,conserv:333:c008706:RH | conserv:333:c008706:T...,conserv:333:c008706:RH | conserv:333:c008706:T...,RH | Temperature,1764187935,1764791019,True,69.350006,48.250000
2,Conserv,conserv:333:c008706,BYCBA_0400410__N____,conserv:333:c008706:RH | conserv:333:c008706:T...,conserv:333:c008706:RH | conserv:333:c008706:T...,RH | Temperature,1764188835,1764791019,True,69.332001,48.279999
3,Conserv,conserv:333:c008706,BYCBA_0400410__N____,conserv:333:c008706:RH | conserv:333:c008706:T...,conserv:333:c008706:RH | conserv:333:c008706:T...,RH | Temperature,1764189735,1764791019,True,69.296005,48.439999
4,Conserv,conserv:333:c008706,BYCBA_0400410__N____,conserv:333:c008706:RH | conserv:333:c008706:T...,conserv:333:c008706:RH | conserv:333:c008706:T...,RH | Temperature,1764190635,1764791019,True,69.061996,48.520000


In [19]:
duckdb.sql("""SELECT DISTINCT
    sr.Source,
    u.datetime_est
FROM '../data/sensor_readings.parquet' sr
LEFT JOIN '../data/utcs.parquet' u 
    ON sr.SensorReadingUTC = u.utc
WHERE sr.Source = 'LI-COR'
ORDER BY sr.SensorReadingUTC
""").to_df()

,Source,datetime_est
0,LI-COR,2025-11-26 05:45:00-07:00
1,LI-COR,2025-11-26 06:00:00-07:00
2,LI-COR,2025-11-26 06:15:00-07:00
3,LI-COR,2025-11-26 06:30:00-07:00
4,LI-COR,2025-11-26 06:45:00-07:00
...,...,...
664,LI-COR,2025-12-03 03:45:00-07:00
665,LI-COR,2025-12-03 04:00:00-07:00
666,LI-COR,2025-12-03 04:15:00-07:00
667,LI-COR,2025-12-03 04:30:00-07:00


# Validation & Alerts

There are two diagnostic files we can review to see if there are alerts or errors. 

In [20]:
# Read ../data/validation-results.csv
import pandas as pd
validation_results = pd.read_csv('../data/validation-results.csv')
validation_results

,run_datetime_est,run_utc,test_name,result,details
0,2025-12-03 14:45:46 EST,1764791146,required_columns_present,PASS,All 11 required columns are present in sensor ...
1,2025-12-03 14:45:46 EST,1764791146,column_data_types,PASS,All columns have the expected data types (e.g....
2,2025-12-03 14:45:46 EST,1764791146,non_null_values,PASS,Every sensor reading row has at least one non-...
3,2025-12-03 14:45:46 EST,1764791146,no_duplicate_readings,PASS,No duplicate readings found. Each sensor has u...
4,2025-12-03 14:45:46 EST,1764791146,sensor_name_consistency,PASS,All sensors have consistent names across all t...
5,2025-12-03 14:45:46 EST,1764791146,reading_interval_check,PASS,All consecutive readings are within 15 minutes...
6,2025-12-03 14:45:46 EST,1764791146,building_info_present,WARN,10 devices have names that could not be parsed...
7,2025-12-03 14:45:46 EST,1764791146,data_gaps_Conserv,WARN,Found 142 gaps in Conserv data where readings ...
8,2025-12-03 14:45:46 EST,1764791146,alerts_Conserv,PASS,No alerts triggered for Conserv. All sensor re...
9,2025-12-03 14:45:46 EST,1764791146,alerts_LI-COR,PASS,No alerts triggered for LI-COR. All sensor rea...


In [21]:
# Validation results that did not pass.
validation_results[validation_results['result'] != "PASS"]

,run_datetime_est,run_utc,test_name,result,details
6,2025-12-03 14:45:46 EST,1764791146,building_info_present,WARN,10 devices have names that could not be parsed...
7,2025-12-03 14:45:46 EST,1764791146,data_gaps_Conserv,WARN,Found 142 gaps in Conserv data where readings ...


In [22]:
# Read ../data/alerts.csv
alerts = pd.read_csv('../data/alerts.csv')
alerts.sample(10).sort_values(by='event_utc')

,event,Source,SensorID,SensorName,event_utc,event_datetime_est,event_end_utc,event_end_datetime_est,gap_minutes,reading_type,reading_value,threshold_min,threshold_max,detected_utc,detected_datetime_est
50,DATA_GAP,Conserv,conserv:333:c008924:Temperature,conserv:333:c008924:Temperature,1764232478,2025-11-27 03:34:38 EST,1764234278,2025-11-27 04:04:38 EST,30.0,NaN,NaN,NaN,NaN,1764791146,2025-12-03 14:45:46 EST
45,DATA_GAP,Conserv,conserv:333:c008924:RH,conserv:333:c008924:RH,1764276577,2025-11-27 15:49:37 EST,1764278377,2025-11-27 16:19:37 EST,30.0,NaN,NaN,NaN,NaN,1764791146,2025-12-03 14:45:46 EST
46,DATA_GAP,Conserv,conserv:333:c008924:RH,conserv:333:c008924:RH,1764299077,2025-11-27 22:04:37 EST,1764300877,2025-11-27 22:34:37 EST,30.0,NaN,NaN,NaN,NaN,1764791146,2025-12-03 14:45:46 EST
128,DATA_GAP,Conserv,conserv:333:c009081:RH,conserv:333:c009081:RH,1764424980,2025-11-29 09:03:00 EST,1764426781,2025-11-29 09:33:01 EST,30.0,NaN,NaN,NaN,NaN,1764791146,2025-12-03 14:45:46 EST
103,DATA_GAP,Conserv,conserv:333:c009069:RH,conserv:333:c009069:RH,1764453242,2025-11-29 16:54:02 EST,1764455042,2025-11-29 17:24:02 EST,30.0,NaN,NaN,NaN,NaN,1764791146,2025-12-03 14:45:46 EST
31,DATA_GAP,Conserv,conserv:333:c008785:RH,conserv:333:c008785:RH,1764459065,2025-11-29 18:31:05 EST,1764462665,2025-11-29 19:31:05 EST,60.0,NaN,NaN,NaN,NaN,1764791146,2025-12-03 14:45:46 EST
59,DATA_GAP,Conserv,conserv:333:c008949:Temperature,conserv:333:c008949:Temperature,1764534131,2025-11-30 15:22:11 EST,1764535931,2025-11-30 15:52:11 EST,30.0,NaN,NaN,NaN,NaN,1764791146,2025-12-03 14:45:46 EST
133,DATA_GAP,Conserv,conserv:333:c009081:Temperature,conserv:333:c009081:Temperature,1764616678,2025-12-01 14:17:58 EST,1764618477,2025-12-01 14:47:57 EST,30.0,NaN,NaN,NaN,NaN,1764791146,2025-12-03 14:45:46 EST
122,DATA_GAP,Conserv,conserv:333:c009072:Temperature,conserv:333:c009072:Temperature,1764642075,2025-12-01 21:21:15 EST,1764643875,2025-12-01 21:51:15 EST,30.0,NaN,NaN,NaN,NaN,1764791146,2025-12-03 14:45:46 EST
137,DATA_GAP,Conserv,conserv:333:c009096:RH,conserv:333:c009096:RH,1764649938,2025-12-01 23:32:18 EST,1764651738,2025-12-02 00:02:18 EST,30.0,NaN,NaN,NaN,NaN,1764791146,2025-12-03 14:45:46 EST


In [23]:
# Group alerts by Source and event.
alerts.groupby(['Source', 'event']).size().reset_index(name='count')

# No alerts (only data gaps) means all readings were within thresholds.

,Source,event,count
0,Conserv,DATA_GAP,142


Now you are ready to move onto analysis to get human-readable results (not indexed by UTC timestamps). See 2-examples-analysis.ipynb.